# 00 — Set up GitHub + Google Drive + inspect PhysioNet EEG

**Goal for the first session:** synchronize your VS Code/GitHub repository into the ephemeral Colab machine, inspect the ZIP stored on Google Drive, extract just `S001R04.edf`, and plot two EEG channels. This notebook deliberately **does not** train a model.

> **How to use:** Open this notebook in **Google Colab**. Edit the two paths in the configuration cell. Execute cells from top to bottom. The first Drive mount requires authorization. On each new Colab runtime you will need to mount Drive and clone the repo again because `/content` is temporary.

**Project workflow:** VS Code → Git commit/push → GitHub → run this notebook in Colab → `git clone` / `git pull` → access the original ZIP from Google Drive.


## 1 — Configuration: edit these two values

In [1]:
from getpass import getpass
from pathlib import Path
import base64
import subprocess
import os

REPO_URL = "https://github.com/andrew2719/cbramod-head-interp.git"


# Replace this with the EXACT path of your downloaded ZIP in Google Drive.
# You can locate it in Colab's Files sidebar after Drive is mounted.
ZIP_PATH = Path("/content/drive/MyDrive/CBraMod-MI/eeg-motor-movementimagery-dataset-1.0.0.zip")

# Working code lives in the temporary Colab runtime; raw data stays on Drive.
REPO_DIR = Path("/content/cbramod-head-interp")
DATA_DIR = Path("/content/data")

print("Repository:", REPO_URL)
print("Drive ZIP:", ZIP_PATH)


Repository: https://github.com/andrew2719/cbramod-head-interp.git
Drive ZIP: /content/drive/MyDrive/CBraMod-MI/eeg-motor-movementimagery-dataset-1.0.0.zip


## 2 — Mount Google Drive

This runs **inside a Colab notebook**, not in a separate shell process.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

print("ZIP exists:", ZIP_PATH.is_file())
if not ZIP_PATH.is_file():
    print("Check ZIP_PATH above; browse the mounted Drive in the Files panel.")


Mounted at /content/drive
ZIP exists: True


In [3]:
token = getpass("GitHub token: ")
encoded = base64.b64encode(f"andrew2719:{token}".encode()).decode()
header = f"http.extraHeader=Authorization: Basic {encoded}"

if (REPO_DIR / ".git").exists():
    command = [
        "git", "-c", header,
        "-C", str(REPO_DIR),
        "pull", "--ff-only",
    ]
else:
    command = [
        "git", "-c", header,
        "clone", REPO_URL, str(REPO_DIR),
    ]

result = subprocess.run(command, capture_output=True, text=True)

# Remove credentials before handling errors.
del token, encoded, header, command

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Git synchronization failed")

print(result.stdout.strip() or "Repository is already current.")

os.chdir(REPO_DIR)
print("Working from:", Path.cwd())

Repository is already current.
Working from: /content/cbramod-head-interp


In [4]:
!pip install mne --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.2 which is incompatible.


In [5]:
# Shared-session hand-off. Notebook 02 validates this before it starts preprocessing.
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f'Dataset not found: {ZIP_PATH}')
if not (REPO_DIR / '.git').is_dir():
    raise RuntimeError('Project Git checkout missing; run the GitHub synchronization cell above.')

CBRAMOD_SETUP_CONTEXT = {
    'drive_root': str(ZIP_PATH.parent),
    'zip_path': str(ZIP_PATH),
    'project_repo_dir': str(REPO_DIR.resolve()),
    'project_repo_url': REPO_URL,
}
CBRAMOD_SETUP_READY = True
print('Setup complete. Ready for notebook 02 on this SAME Colab runtime.')
print('Project code:', CBRAMOD_SETUP_CONTEXT['project_repo_dir'])
print('Input ZIP:', CBRAMOD_SETUP_CONTEXT['zip_path'])
print('Training outputs: MyDrive/CBraMod-MI/results/physionet_mi_4class/')


Setup complete. Ready for notebook 02 on this SAME Colab runtime.
Project code: /content/cbramod-head-interp
Input ZIP: /content/drive/MyDrive/CBraMod-MI/eeg-motor-movementimagery-dataset-1.0.0.zip
Training outputs: MyDrive/CBraMod-MI/results/physionet_mi_4class/


## 8 — Record what you learned

Answer these in your project README or lab journal:

1. How many electrodes and time samples are in `S001R04.edf`?
2. What does the sampling rate tell you?
3. What do `T0`, `T1`, and `T2` mean **in run 04**?
4. Why is a 4-second window from a 160 Hz recording represented by `(64, 640)`?
5. Why do we keep raw EDF files in Drive or `/content/eeg_data`, outside the Git repo?

**Next notebook:** pick event-aligned epochs for left/right imagery, study filtering and resampling, and then build a simple decoding baseline. Do not start interpreting attention weights until you have a trained classifier and a held-out evaluation set.
